# 04; Interpretability (calm regime)

Does the functional form of crypto market impact match the equity model, or does the data support a different structure? Does the answer change across market regimes?

This notebook addresses the first part (calm regime) by extracting a closed-form expression from MLP-A via PySR and comparing it directly against the theoretical predictions.

**4a; Sensitivity analysis**
Hold all inputs at their training median. Sweep each input across its
empirical range and record the model output.

**4b; Symbolic regression with PySR**
Feed MLP-A input/output pairs into PySR and extract a closed-form
expression. Compare to:
- Theoretical sqrt law: I ~ Q^0.5
- OLS benchmark: delta = 0.107
- What PySR actually finds

MLP-A is used throughout because its inputs match the OLS benchmarks
exactly (log_Q, log_sigma, log_V). MLP-B has extra features that would
complicate the comparison.

Three outcomes are possible going in:
- PySR finds exponent ~ 0.5: the MLP recovered the equity sqrt law
- PySR finds exponent ~ 0.1: the MLP learned the same shallow empirical relationship as OLS
- PySR finds a non-power-law form: the crypto data has a different structure entirely

All three are informative.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import joblib
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import sys
sys.path.append("../src")

DATA_DIR    = Path("../data/processed")
OUTPUT_DIR  = Path("../outputs")
MODEL_DIR   = OUTPUT_DIR / "models"
FIGURES_DIR = OUTPUT_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

torch.manual_seed(42)
np.random.seed(42)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

## Load model and data

In [ ]:
# Same architecture as 03_mlp.ipynb
class ImpactMLP(nn.Module):
    def __init__(self, input_dim: int, hidden: int = 64, layers: int = 3, dropout: float = 0.3):
        super().__init__()
        net = [nn.Linear(input_dim, hidden), nn.ReLU(), nn.Dropout(dropout)]
        for _ in range(layers - 1):
            net += [nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout)]
        net += [nn.Linear(hidden, 1)]
        self.net = nn.Sequential(*net)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)


FEATURES_A = ["log_Q", "log_sigma", "log_V"]

scaler_a = joblib.load(MODEL_DIR / "scaler_a.pkl")

mlp_a = ImpactMLP(input_dim=len(FEATURES_A)).to(DEVICE)
mlp_a.load_state_dict(torch.load(MODEL_DIR / "mlp_a.pt", map_location=DEVICE))
mlp_a.eval()

train = pd.read_parquet(DATA_DIR / "mo_train.parquet")
test  = pd.read_parquet(DATA_DIR / "mo_test.parquet")

def prepare(df: pd.DataFrame) -> pd.DataFrame:
    df = df[(df["I"] > 0) & (df["Q_norm"] > 0) & (df["sigma_daily"] > 0)].copy()
    df["log_I"]     = np.log(df["I"])
    df["log_Q"]     = np.log(df["Q_norm"])
    df["log_sigma"] = np.log(df["sigma_daily"])
    df["log_V"]     = np.log(df["V_daily"])
    mu, sd = df["log_I"].mean(), df["log_I"].std()
    df = df[np.abs(df["log_I"] - mu) < 3 * sd].reset_index(drop=True)
    return df

train_p = prepare(train)
test_p  = prepare(test)

print(f"Train: {len(train_p):,}")
print(f"Test : {len(test_p):,}")

## Helper

In [ ]:
def predict(model: ImpactMLP, X: np.ndarray) -> np.ndarray:
    model.eval()
    with torch.no_grad():
        t = torch.tensor(X, dtype=torch.float32).to(DEVICE)
        return model(t).cpu().numpy()

---
## 4a; Sensitivity analysis

For each feature: hold all others at training median, sweep the target
feature across its 1st-99th percentile range, record predicted log_I.

The log_Q sweep is the key one. In a pure power law the curve is linear
in log-log space with slope = delta. Deviation from linearity means the
network learned something more complex.

In [ ]:
medians = train_p[FEATURES_A].median().values
p01     = train_p[FEATURES_A].quantile(0.01).values
p99     = train_p[FEATURES_A].quantile(0.99).values

N_SWEEP = 200

sensitivity = {}

for i, feature in enumerate(FEATURES_A):
    sweep = np.linspace(p01[i], p99[i], N_SWEEP)

    # All features at median except the one being swept
    X_sweep = np.tile(medians, (N_SWEEP, 1))
    X_sweep[:, i] = sweep

    X_scaled = scaler_a.transform(X_sweep)
    y_pred   = predict(mlp_a, X_scaled)

    sensitivity[feature] = {"sweep": sweep, "pred": y_pred}
    print(f"{feature}: range [{p01[i]:.3f}, {p99[i]:.3f}]  pred range [{y_pred.min():.3f}, {y_pred.max():.3f}]")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

labels = {
    "log_Q":     "log(Q_norm); participation rate",
    "log_sigma": "log(sigma_daily); daily realized vol",
    "log_V":     "log(V_daily); daily volume",
}

for ax, feature in zip(axes, FEATURES_A):
    s = sensitivity[feature]
    ax.plot(s["sweep"], s["pred"], color="steelblue", linewidth=2)
    ax.axvline(medians[FEATURES_A.index(feature)], color="gray",
               linestyle="--", linewidth=1, label="Median")
    ax.set_xlabel(labels[feature])
    ax.set_ylabel("Predicted log I")
    ax.set_title(f"Sensitivity: {feature}")
    ax.legend()

plt.suptitle("MLP-A sensitivity analysis; calm regime", y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "sensitivity_calm.png", bbox_inches="tight")
plt.show()

### Local slope of the log_Q curve

In a pure power law I ~ Q^delta, d(log I)/d(log Q) = delta everywhere.
A constant slope means the network learned a power law.
A varying slope means it learned something more complex.

A negative slope means larger metaorders have lower normalized impact,
which contradicts standard theory. Flagged explicitly if observed.

In [ ]:
s = sensitivity["log_Q"]

# Numerical derivative d(log I) / d(log Q)
# Equals delta everywhere if the network learned a power law
dlog_I_dlog_Q = np.gradient(s["pred"], s["sweep"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(s["sweep"], s["pred"], color="steelblue", linewidth=2)
axes[0].set_xlabel("log(Q_norm)")
axes[0].set_ylabel("Predicted log I")
axes[0].set_title("log_Q sensitivity curve")

axes[1].plot(s["sweep"], dlog_I_dlog_Q, color="firebrick", linewidth=2,
             label="d(log I)/d(log Q)")
axes[1].axhline(0.5,   color="gray",      linestyle="--", linewidth=1.5, label="Sqrt law (0.5)")
axes[1].axhline(0.107, color="steelblue", linestyle=":",  linewidth=1.5, label="OLS delta (0.107)")
axes[1].axhline(0.0,   color="black",     linestyle="-",  linewidth=0.8, alpha=0.4)
axes[1].set_xlabel("log(Q_norm)")
axes[1].set_ylabel("Local slope")
axes[1].set_title("Local power-law exponent - MLP-A")
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "local_slope_calm.png", bbox_inches="tight")
plt.show()

print(f"Mean local slope : {dlog_I_dlog_Q.mean():.4f}")
print(f"Slope range      : [{dlog_I_dlog_Q.min():.4f}, {dlog_I_dlog_Q.max():.4f}]")
print(f"Slope std        : {dlog_I_dlog_Q.std():.4f}")

if dlog_I_dlog_Q.mean() < 0:
    print("Note: mean slope is negative.")
    print("The network learned that larger metaorders have lower normalized impact.")
    print("This contradicts standard theory. Three possible explanations:")
    print("  1. Reconstruction artifact: synthetic sizes may not reflect true")
    print("     institutional size distribution in a retail-dominated crypto market.")
    print("  2. Large Q_norm in calm conditions may signal noise trading rather")
    print("     than informed flow, inverting the usual size-impact relationship.")
    print("  3. Normalization confound: Q_norm = Q / V_daily. High-volume days")
    print("     make large absolute trades appear small in normalized terms.")

---
## 4b; Symbolic regression with PySR

PySR fits a closed-form expression to the MLP's input-output pairs.
It targets MLP predictions, not raw log_I, so the result describes
what the network learned rather than re-fitting the data directly.
Features are passed unscaled so the output formula is human-readable.

Sampling: 40K observations stratified across the log_Q range. A uniform
draw would oversample the median and underrepresent large metaorders.

Formula selection: elbow method on the Pareto front with a minimum
complexity of 5. Formulas below that threshold tend to ignore log_Q
entirely, which defeats the purpose of the analysis.

In [ ]:
from pysr import PySRRegressor

N_PYSR      = 40_000
N_BINS_PYSR = 20
PER_BIN     = N_PYSR // N_BINS_PYSR

# Stratified sample across log_Q range
train_pysr = train_p.copy()
train_pysr["q_bin"] = pd.qcut(train_pysr["log_Q"], q=N_BINS_PYSR, duplicates="drop")

frames = []
for _, grp in train_pysr.groupby("q_bin", observed=True):
    n = min(PER_BIN, len(grp))
    frames.append(grp.sample(n=n, random_state=42))

sample = pd.concat(frames).reset_index(drop=True)
print(f"PySR sample size : {len(sample):,}")
print(f"log_Q range      : [{sample['log_Q'].min():.2f}, {sample['log_Q'].max():.2f}]")

X_pysr_raw = sample[FEATURES_A].values
X_pysr     = scaler_a.transform(X_pysr_raw)
y_pysr     = predict(mlp_a, X_pysr)  # MLP predictions, not raw log_I

print(f"y_pysr range     : [{y_pysr.min():.3f}, {y_pysr.max():.3f}]")

In [ ]:
# Fit on unscaled features so the extracted formula is interpretable
model_pysr = PySRRegressor(
    niterations        = 50,
    binary_operators   = ["+", "-", "*", "/", "^"],
    unary_operators    = ["log", "exp", "sqrt", "abs"],
    constraints        = {"^": (-1, 1)},
    maxsize            = 20,
    populations        = 30,
    population_size    = 50,
    parsimony          = 0.01,
    random_state       = 42,
    verbosity          = 1,
    temp_equation_file = True,
)

model_pysr.fit(X_pysr_raw, y_pysr, variable_names=FEATURES_A)

In [ ]:
print(model_pysr)
print(model_pysr.get_best())

### Select formula from the Pareto front

Elbow method with minimum complexity of 5. The elbow is the point where
adding more complexity stops producing meaningful loss reduction.

In [ ]:
equations = model_pysr.equations_.copy()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(equations["complexity"], equations["loss"],
        marker="o", color="steelblue", linewidth=1.5)
for _, row in equations.iterrows():
    label = str(row["equation"])[:30]
    ax.annotate(label, (row["complexity"], row["loss"]),
                textcoords="offset points", xytext=(5, 0),
                fontsize=6, alpha=0.7)
ax.set_xlabel("Complexity")
ax.set_ylabel("Loss (MSE on PySR sample)")
ax.set_title("PySR Pareto front - calm regime")
ax.set_yscale("log")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "pysr_pareto_calm.png", bbox_inches="tight")
plt.show()

In [ ]:
MIN_COMPLEXITY = 5

equations  = equations.sort_values("complexity").reset_index(drop=True)
candidates = equations[equations["complexity"] >= MIN_COMPLEXITY].reset_index(drop=True)

if len(candidates) < 2:
    elbow_idx_in_candidates = 0
else:
    loss_drop  = candidates["loss"].values[:-1] - candidates["loss"].values[1:]
    efficiency = loss_drop / np.ones(len(loss_drop))
    elbow_idx_in_candidates = int(np.argmax(efficiency)) + 1

best_eq   = candidates.iloc[elbow_idx_in_candidates]
elbow_idx = best_eq.name

print(f"Selected (elbow, complexity >= {MIN_COMPLEXITY}):")
print(f"  Complexity : {best_eq['complexity']}")
print(f"  Loss       : {best_eq['loss']:.6f}")
print(f"  Equation   : {best_eq['equation']}")

pysr_best = model_pysr.get_best()
print(f"  Complexity : {pysr_best['complexity']}")
print(f"  Loss       : {pysr_best['loss']:.6f}")
print(f"  Equation   : {pysr_best['equation']}")

### Interpret the selected formula

Structure: `(A + B * log_Q) / log_sigma`

1. The 1/log_sigma term dominates. Impact falls as volatility rises.
   Almgren-Chriss predicts the opposite sign (beta = +1).
   sigma_daily is near-categorical (one value per day), so this likely
   reflects day-level variation rather than a true vol-impact relationship.

2. The log_Q coefficient is small. Its sign determines whether the network
   learned a positive or negative size-impact relationship.

3. No Q^0.5 structure. The sqrt law is absent.

### How well does the PySR formula approximate the MLP?

In [ ]:
X_te_raw  = test_p[FEATURES_A].values
X_te      = scaler_a.transform(X_te_raw)
y_te_mlp  = predict(mlp_a, X_te)
y_te_true = test_p["log_I"].values

# PySR applied to unscaled features
y_te_pysr = model_pysr.predict(X_te_raw, index=elbow_idx)

print("PySR vs MLP predictions (test set):")
print(f"  MSE : {mean_squared_error(y_te_mlp, y_te_pysr):.4f}")
print(f"  R2  : {r2_score(y_te_mlp, y_te_pysr):.4f}")
print()
print("PySR vs actual log_I:")
print(f"  MSE : {mean_squared_error(y_te_true, y_te_pysr):.4f}")
print(f"  R2  : {r2_score(y_te_true, y_te_pysr):.4f}")
print()
print("MLP-A vs actual log_I (reference from notebook 03):")
print(f"  MSE : {mean_squared_error(y_te_true, y_te_mlp):.4f}")
print(f"  R2  : {r2_score(y_te_true, y_te_mlp):.4f}")

### Visual comparison: MLP vs PySR vs OLS vs sqrt law

In [ ]:
# Sweep log_Q with sigma and V held at training medians
median_sigma = train_p["log_sigma"].median()
median_V     = train_p["log_V"].median()

q_range = np.linspace(
    train_p["log_Q"].quantile(0.01),
    train_p["log_Q"].quantile(0.99),
    200
)

X_line_raw = np.column_stack([
    q_range,
    np.full_like(q_range, median_sigma),
    np.full_like(q_range, median_V),
])

y_mlp_line  = predict(mlp_a, scaler_a.transform(X_line_raw))
y_pysr_line = model_pysr.predict(X_line_raw, index=elbow_idx)

# OLS power law from 02_benchmark.ipynb
OLS_DELTA     = 0.107
OLS_INTERCEPT = train_p["log_I"].mean() - OLS_DELTA * train_p["log_Q"].mean()
y_ols_line    = OLS_INTERCEPT + OLS_DELTA * q_range
y_sqrt_line   = OLS_INTERCEPT + 0.5 * q_range

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(q_range, y_mlp_line,  color="steelblue", linewidth=2,
        label="MLP-A")
ax.plot(q_range, y_pysr_line, color="darkorange", linewidth=2,
        label=f"PySR (complexity {best_eq['complexity']})")
ax.plot(q_range, y_ols_line,  color="firebrick",  linewidth=1.5, linestyle="--",
        label=f"OLS power law (delta={OLS_DELTA})")
ax.plot(q_range, y_sqrt_line, color="gray",       linewidth=1.5, linestyle=":",
        label="Sqrt law (delta=0.5)")

ax.set_xlabel("log(Q_norm)")
ax.set_ylabel("Predicted log I")
ax.set_title("Impact formula comparison - calm regime\n(sigma and V at training median)")
ax.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "formula_comparison_calm.png", bbox_inches="tight")
plt.show()

### Stability check

Three random subsamples of 10K observations, different seeds.
If the same structural form appears each time, the result is not
a search artifact. If the structure changes across seeds, treat
the PySR output with caution.

In [ ]:
STABILITY_SEEDS = [0, 1, 2]
STABILITY_N     = 10_000

for seed in STABILITY_SEEDS:
    rng_s = np.random.default_rng(seed)
    idx   = rng_s.choice(len(X_pysr_raw), size=STABILITY_N, replace=False)
    X_sub = X_pysr_raw[idx]
    y_sub = predict(mlp_a, scaler_a.transform(X_sub))

    m = PySRRegressor(
        niterations        = 30,
        binary_operators   = ["+", "-", "*", "/", "^"],
        unary_operators    = ["log", "exp", "sqrt", "abs"],
        constraints        = {"^": (-1, 1)},
        maxsize            = 15,
        populations        = 20,
        population_size    = 40,
        parsimony          = 0.01,
        random_state       = seed,
        verbosity          = 0,
        temp_equation_file = True,
    )
    m.fit(X_sub, y_sub, variable_names=FEATURES_A)

    best = m.get_best()
    print(f"Seed {seed}: complexity={best['complexity']}  loss={best['loss']:.4f}  eq={best['equation']}")

---
## Summary

In [ ]:
calm_summary = {
    "ols_delta":       float(OLS_DELTA),
    "mlp_mean_slope":  float(dlog_I_dlog_Q.mean()),
    "pysr_complexity": int(best_eq["complexity"]),
    "pysr_loss":       float(best_eq["loss"]),
    "pysr_equation":   str(best_eq["equation"]),
}
pd.Series(calm_summary).to_json(OUTPUT_DIR / "calm_summary.json")

joblib.dump(model_pysr, MODEL_DIR / "pysr_calm.pkl")
joblib.dump(elbow_idx,  MODEL_DIR / "pysr_calm_elbow_idx.pkl")

print(f"  Mean local slope : {dlog_I_dlog_Q.mean():.4f}")
print(f"  Slope range      : [{dlog_I_dlog_Q.min():.4f}, {dlog_I_dlog_Q.max():.4f}]")
print(f"  Slope std        : {dlog_I_dlog_Q.std():.4f}")
print(f"PySR formula (complexity >= {MIN_COMPLEXITY}, elbow):")
print(f"  Complexity : {best_eq['complexity']}")
print(f"  Loss       : {best_eq['loss']:.6f}")
print(f"  Equation   : {best_eq['equation']}")
print(f"PySR approximation quality (test set):")
print(f"  R2 (PySR vs MLP)   : {r2_score(y_te_mlp, y_te_pysr):.4f}")
print(f"  R2 (PySR vs log_I) : {r2_score(y_te_true, y_te_pysr):.4f}")
print(f"  Sqrt law delta  : 0.500")
print(f"  OLS delta       : {OLS_DELTA}")
print(f"  MLP mean slope  : {dlog_I_dlog_Q.mean():.4f}")